<a href="https://colab.research.google.com/github/trankhactue/fall/blob/main/trankhactue-202625099Untitled5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

import warnings
warnings.filterwarnings("ignore")
sns.set(style="darkgrid")

SEED = 42
np.random.seed(SEED)

# 데이터 로드 및 탐색
raw_path = "https://raw.githubusercontent.com/wesm/pydata-book/3rd-edition/datasets/titanic/"
df_train = pd.read_csv(f"{raw_path}train.csv")
df_test = pd.read_csv(f"{raw_path}test.csv")

print(f"Train 크기: {df_train.shape}, Test 크기: {df_test.shape}")
display(df_train.head())
df_train.info()

# 결측치 현황 확인
missing_df = pd.DataFrame({
    "결측치_수": df_train.isna().sum(),
    "결측치_비율(%)": (df_train.isna().mean() * 100).round(2)
})
print("\n[1. 결측치 현황]")
display(missing_df[missing_df["결측치_수"] > 0])

# 타겟 변수 분포 확인
print("\n[Survived 분포]")
print(df_train["Survived"].value_counts())
print((df_train["Survived"].value_counts(normalize=True) * 100).round(2))

In [ ]:
def extract_title(name: str) -> str:
    title = name.split(",")[1].split(".")[0].strip()
    female_titles = {"Miss", "Mrs", "Ms", "Mlle", "Lady", "Mme", "the Countess", "Dona"}
    special_male = {"Dr", "Col", "Major", "Jonkheer", "Capt", "Sir", "Don", "Rev"}
    if title in female_titles:
        return "Female_Title"
    elif title == "Master":
        return "Master"
    elif title in special_male:
        return "Special_Male"
    else:
        return "Mr"

def categorize_family(size: int) -> str:
    if size == 1:
        return "Alone"
    elif 2 <= size <= 4:
        return "Small"
    elif 5 <= size <= 6:
        return "Medium"
    else:
        return "Large"

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    data["Cabin_Has"] = data["Cabin"].notna().astype(int)
    data = data.drop(columns=["Cabin"])
    data["Family_Size"] = data["SibSp"] + data["Parch"] + 1
    data["Family_Group"] = data["Family_Size"].apply(categorize_family)
    data["Title"] = data["Name"].apply(extract_title)
    drop_cols = [c for c in ["PassengerId", "Name", "Ticket"] if c in data.columns]
    return data.drop(columns=drop_cols)

train_fe = engineer_features(df_train)
test_fe = engineer_features(df_test)

X = train_fe.drop(columns=["Survived"])
y = train_fe["Survived"].astype(int)
X_test_final = test_fe.copy()

print("[2. Feature Engineering 결과 - 파생변수 확인]")
display(X[["Family_Size", "Family_Group", "Title", "Cabin_Has"]].head())

# 인코딩 결과 확인 (OneHotEncoder 미리보기)
preview_cat_cols = ["Sex", "Embarked", "Pclass", "Family_Group", "Title"]
ohe_preview = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_imputed_preview = SimpleImputer(strategy="most_frequent").fit_transform(X[preview_cat_cols])
encoded_arr = ohe_preview.fit_transform(cat_imputed_preview)
encoded_df = pd.DataFrame(encoded_arr, columns=ohe_preview.get_feature_names_out(preview_cat_cols))

print(f"\n[2. 원핫 인코딩 결과 크기]: {encoded_df.shape}")
display(encoded_df.head())

In [ ]:
class GroupAgeImputer(BaseEstimator, TransformerMixin):
    def fit(self, X_df, y=None):
        df = pd.DataFrame(X_df).copy()
        self.global_median_ = df["Age"].median()
        self.group_medians_ = df.groupby(["Pclass", "Sex"])["Age"].median()
        return self

    def transform(self, X_df):
        df = pd.DataFrame(X_df).copy()
        for (pclass, sex), med in self.group_medians_.items():
            mask = df["Age"].isna() & (df["Pclass"] == pclass) & (df["Sex"] == sex)
            df.loc[mask, "Age"] = med
        df["Age"] = df["Age"].fillna(self.global_median_)
        return df[["Age"]].to_numpy()

age_cols = ["Age", "Pclass", "Sex"]
num_cols = ["Fare", "SibSp", "Parch", "Family_Size", "Cabin_Has"]
cat_cols = ["Sex", "Embarked", "Pclass", "Family_Group", "Title"]

preprocessor = ColumnTransformer([
    ("age", Pipeline([("group_age", GroupAgeImputer()), ("scaler", StandardScaler())]), age_cols),
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols)
], remainder="drop")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

base_knn_pipe = Pipeline([
    ("prep", preprocessor),
    ("knn", KNeighborsClassifier())
])

base_scores = cross_val_score(base_knn_pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
print("[3. 기본 KNN 5-Fold Stratified 교차검증 결과]")
for idx, score in enumerate(base_scores, 1):
    print(f"  - Fold {idx} Accuracy: {score:.4f}")
print(f"  - 평균 Accuracy: {base_scores.mean():.4f} (+/- {base_scores.std():.4f})")

In [ ]:
param_grid = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]
}

grid_search = GridSearchCV(
    estimator=base_knn_pipe,
    param_grid=param_grid,
    cv=skf,
    scoring="accuracy",
    n_jobs=-1,
    refit=True
)

grid_search.fit(X, y)

best_params = grid_search.best_params_
best_score = grid_search.best_score_

print("[4. GridSearchCV 최적 하이퍼파라미터 탐색 결과]")
print(f"  - 최적 하이퍼파라미터: {best_params}")
print(f"  - 최적 5-Fold 평균 Accuracy: {best_score:.4f}")

In [ ]:
diff_acc = best_score - base_scores.mean()

comparison_df = pd.DataFrame({
    "구분": ["튜닝 전 (기본 KNN)", "튜닝 후 (최적 KNN)"],
    "5-Fold 평균 Accuracy": [round(base_scores.mean(), 4), round(best_score, 4)],
    "하이퍼파라미터": ["n_neighbors=5, weights='uniform', p=2", str(best_params)]
})

print("[5. 튜닝 전후 Accuracy 비교]")
display(comparison_df)
print(f"Accuracy 변화량(튜닝 후 - 튜닝 전): {diff_acc:+.4f}")

최종 결과에 대한 간단한 해석 (3~5줄)
6. 최종 결과 해석

Age 결측치를 Pclass와 Sex 그룹별 중앙값으로 보완하고 Family_Size, Family_Group, Title, Cabin_Has 파생변수를 생성한 뒤 원핫 인코딩과 StandardScaler를 적용해 거리 기반 KNN의 스케일 차이를 줄였습니다.

전처리와 스케일링을 Pipeline 내부에 포함함으로써 5-Fold Stratified 교차검증 과정에서 발생할 수 있는 데이터 누수(Data Leakage)를 방지했습니다.

GridSearchCV로 n_neighbors, weights, p 조합을 탐색한 결과 기본 KNN 모델 대비 평균 Accuracy가 개선되어, 데이터 특성에 맞는 이웃 수와 거리 가중치(weights) 설정이 분류 성능 향상에 유효함을 확인했습니다.